In [1]:
%pip install -q "transformers==4.57.1" "accelerate==1.11.0" pillow pandas requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import json
import math
import re
from pathlib import Path

import requests
import torch
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display
from tqdm.auto import tqdm
from transformers import (
    AutoProcessor, GenerationConfig,
    Qwen3VLForConditionalGeneration
)

In [5]:
ROOT = Path("/content/drive/MyDrive/MLLM/mordern_llm")
DATA = ROOT / "data/grounding20"
IMAGES = DATA / "images"
OUTPUTS = ROOT / "outputs/grounding"
MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"

IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=False)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
    attn_implementation="sdpa"
).eval()

GEN = GenerationConfig(
    max_new_tokens=64,
    do_sample=False,
    bos_token_id=model.generation_config.bos_token_id,
    eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=processor.tokenizer.pad_token_id
)

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

In [6]:
source_path = DATA / "source_rows.json"

if not source_path.exists():
    response = requests.get(
        "https://datasets-server.huggingface.co/rows",
        params={
            "dataset": "lmms-lab-encoder/RefCOCO",
            "config": "default",
            "split": "val",
            "offset": 0,
            "length": 20
        },
        timeout=30
    )
    response.raise_for_status()
    source_path.write_text(
        json.dumps(response.json(), ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

source = json.loads(source_path.read_text(encoding="utf-8"))
print(len(source))

5


In [7]:
examples = []
for item in source["rows"]:
    row = item["row"]
    match = re.match(
        r"(COCO_(train2014|val2014)_\d{12})",
        row["file_name"]   
    )
    
    filename = match.group(1) + ".jpg"
    split_folder = match.group(2)
    image_path = IMAGES / filename
    
    if not image_path.exists():
        url = f"http://images.cocodataset.org/{split_folder}/{filename}"
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        image_path.write_bytes(response.content)
    
    with Image.open(image_path) as image:
        width, height = image.size
    
    x, y, w, h = map(float, row["bbox"])
    examples.append({
        "id": row["question_id"],
        "phrase": row["answer"][0],
        "image_path": str(image_path),
        "width": width,
        "height": height,
        "gt_xyxy": [x, y, x + w, y + h],
    })

(DATA / "manifest.json").write_text(
    json.dumps(examples, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Examples:", len(examples))
print("Unique images:", len({e["image_path"] for e in examples}))
print("First phrase:", examples[0]["phrase"])

Examples: 20
Unique images: 9
First phrase: bowl behind the others can only see part


**Parser, coordinate conversion and IoU**

In [8]:
def parse_box(raw):
    text = raw.strip()
    lines = text.splitlines()
    wrapped = (
        len(lines) >= 3
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    )
    
    if wrapped:
        text = "\n".join(lines[1:-1]).strip()
    
    obj = json.loads(text)
    if not (isinstance(obj, list) and len(obj) == 1 and isinstance(obj[0], dict)):
        raise ValueError("Expected one object in a JSON array")
    
    box = obj[0].get("bbox_2d")
    if not (isinstance(box, list) and len(box) == 4):
        raise ValueError("Expected four coordinates")

    if not all(
        type(v) in (int, float) and math.isfinite(v)
        and 0 <= v <= 1000 for v in box
    ):
        raise ValueError("Coordinates must be finite numbers in 0..1000")
    
    x1, y1, x2, y2 = map(float, box)
    if not (x1 < x2 and y1 < y2):
        raise ValueError("Invalid xyxy order or zero area")

    return [x1, y1, x2, y2], wrapped

In [9]:
def to_pixels(box, width, height):
    return [
        box[0] * width / 1000,
        box[1] * height / 1000,
        box[2] * width / 1000,
        box[3] * height / 1000,
    ]

def box_iou(a, b):
    iw = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = iw * ih
    
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])

    return intersection / (area_a + area_b - intersection)

**Inference**

In [10]:
@torch.inference_mode()
def ground(image_path, phrase):
    with Image.open(image_path) as source:
        image = source.convert("RGB")
        
    question = (
        f"Locate the single object described by: {phrase!r}. "
        'Return a JSON array containing exactly one object: '
        '[{"bbox_2d": [x1, y1, x2, y2]}]. '
        "Use xyxy coordinates normalized to 0..1000 "
        "relative to the whole image. "
        "Output only JSON, without markdown or explanation."
    )
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": question},
        ],
    }]
    
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    batch = processor(
        text=[prompt],
        images=[image],
        images_kwargs={
            "do_resize": True,
            "min_pixels": 64 * 32 * 32,
            "max_pixels": 400 * 32 * 32,
        },
        return_tensors="pt"
    ).to(model.device)
    
    output = model.generate(
        **batch,
        generation_config=GEN,
        use_model_defaults=False,
        do_sample=False,
    )

    new_ids = output[:, batch["input_ids"].shape[1]:]
    return processor.batch_decode(
        new_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

In [11]:
def evaluate_example(example):
    raw = ground(example["image_path"], example["phrase"])
    
    try:
        normalized_box, wrapped = parse_box(raw)
        prediction = to_pixels(
            normalized_box, example["width"], example["height"]
        )
        iou = box_iou(prediction, example["gt_xyxy"])
        error = None
    
    except ValueError as exc:
        prediction = None
        iou = 0.0
        wrapped = False
        error = str(exc)
    
    return {
        **example,
        "raw_answer": raw,
        "pred_xyxy": prediction,
        "used_fence_cleanup": wrapped,
        "box_valid": prediction is not None,
        "iou": iou,
        "success": prediction is not None and iou >= 0.5,
        "error": error
    }

first_result = evaluate_example(examples[0])

print("Phrase:", first_result["phrase"])
print("Raw answer:", first_result["raw_answer"])
print("Pixel prediction:", first_result["pred_xyxy"])
print("IoU:", round(first_result["iou"], 3))

Phrase: bowl behind the others can only see part
Raw answer: ```json
[
	{"bbox_2d": [183, 0, 720, 433], "label": "bowl behind the others can only see part"}
]
```
Pixel prediction: [117.12, 0.0, 460.8, 185.324]
IoU: 0.0
